In [17]:
#Loading packages required
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.metrics import pairwise_distances
from sklearn.manifold import TSNE
import umap
from mpl_toolkits.mplot3d import Axes3D

In [18]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [19]:
!ls "/content/drive"

MyDrive


#Dimensionality Reduction using PCA

In [21]:
meta_data = pd.read_csv("/content/drive/MyDrive/BT 3041 - Course Project Repository/Data/cancer_metadata.tsv", sep = "\t", index_col = 0)

In [22]:
data = pd.read_csv("/content/drive/MyDrive/BT 3041 - Course Project Repository/Data/RNAseq_Tumor_Data.tsv", sep = "\t", index_col = 0)

In [23]:
# Transpose so rows = Samples, columns = Genes and clean the data
X = data.T
#Handle missing values (Fill NaNs with 0)
X = X.fillna(0)
#Remove "Constant" Genes (Genes that have 0 variance)
X = X.loc[:, (X != X.iloc[0]).any()]

gene_names = X.columns

# Standardize the data - Scaling is necessary before doing PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Transposed Data Shape: {X.shape} (Samples, Genes)")

Transposed Data Shape: (6052, 20305) (Samples, Genes)


In [ ]:
labels = []

for col in X.index:
    match = meta_data.loc[
        meta_data["bcr_patient_barcode"] == col[0:12],
        "acronym"
    ]

    if not match.empty:
        labels.append(match.iloc[0])
    else:
        labels.append(None)  # or np.nan if preferred

In [ ]:
# Run PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# --- Scree Plot: Individual Variance Contribution ---
exp_var = pca.explained_variance_ratio_*100

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(exp_var) + 1), exp_var, color='skyblue', edgecolor='navy', alpha=0.8)
plt.title("Individual Variance Explained by Each PC", fontsize=14)
plt.xlabel("Principal Component")
plt.ylabel("Variance Explained (%)")
plt.xticks(range(1, len(exp_var)))
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Cumulative Variance Contribution
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(exp_var) + 1), np.cumsum(exp_var), marker='o', linestyle='-', color='red', linewidth=2)
plt.axhline(y=90, color='gray', linestyle='--') # 90% threshold line
plt.text(1, 92, "90% Threshold", color='gray', fontweight='bold')
plt.title("Cumulative Variance Explained", fontsize=14)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Variance (%)")
plt.xticks(range(1, len(exp_var) + 1))
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
#Score Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue = labels,
    palette='tab10',
    s=120,
    edgecolor='black',
    alpha=0.9
)
plt.axhline(0, color='grey', lw=1, linestyle='--', alpha=0.5)
plt.axvline(0, color='grey', lw=1, linestyle='--', alpha=0.5)
plt.xlabel(f"PC1 ({exp_var[0]:.1f}%)", fontsize=12)
plt.ylabel(f"PC2 ({exp_var[1]:.1f}%)", fontsize=12)
plt.title("PCA Score Plot: Cancer Types", fontsize=14)
plt.legend(title="Cancer Type", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

In [ ]:
#3D score plot
pca_3D = PCA(n_components=3)
X_pca_3D = pca_3D.fit_transform(X_scaled)
exp_var_3D = pca_3D.explained_variance_ratio_ * 100

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
unique_labels = list(set(labels))
color_map = dict(zip(unique_labels, sns.color_palette("tab10", len(unique_labels))))
colors = [color_map[l] for l in labels]


ax.scatter( X_pca_3D[:, 0], X_pca_3D[:, 1], X_pca_3D[:, 2], c=colors, s=100, edgecolor='black', alpha=0.9)

ax.set_xlabel(f"PC1 ({exp_var_3D[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({exp_var_3D[1]:.1f}%)")
ax.set_zlabel(f"PC3 ({exp_var_3D[2]:.1f}%)")
ax.set_title("3D PCA Score Plot: Cancer Types")

legend_elements = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        label=label,
        markerfacecolor=color_map[label],
        markersize=8
    )
    for label in unique_labels
]

plt.legend(title = "Cancer Types",
    handles=legend_elements,
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
plt.tight_layout()
plt.show()

In [ ]:
#Loadings plot
# Loadings (genes × PCs)
loadings = pca.components_.T

#top_n = 20  # number of genes of high importance based on PC1 and PC2
# Combined contribution (distance from origin in PC1–PC2 space)
#gene_contributions = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
#top_genes_idx = np.argsort(gene_contributions)[-top_n:]

top_n = 10
top_pc1 = np.argsort(np.abs(loadings[:, 0]))[-top_n:]
top_pc2 = np.argsort(np.abs(loadings[:, 1]))[-top_n:]

top_genes_idx = np.unique(np.concatenate([top_pc1, top_pc2]))

plt.figure(figsize=(10, 8))
plt.scatter(loadings[:, 0], loadings[:, 1],alpha=0.1, c='gray', s=5)
sorted_indices = top_genes_idx[np.argsort(loadings[top_genes_idx, 1])]
for i, idx in enumerate(sorted_indices):
    x, y = loadings[idx, 0], loadings[idx, 1]
    plt.annotate( '', xy=(x, y), xytext=(0, 0),arrowprops=dict(arrowstyle='->', color='red', lw=1.5, alpha=0.7) )
    stagger_step = 0.001
    y_offset = (y * 1.3) + (i * stagger_step if y > 0 else -i * stagger_step)
    x_offset = x * 1.3
    ha = 'left' if x > 0 else 'right'
    plt.text( x_offset, y_offset, gene_names[idx],fontsize=9, color='darkred', fontweight='bold', ha=ha, va='center')

max_val = np.max(np.abs(loadings[top_genes_idx, :2])) * 3.0
plt.xlim(-max_val, max_val)
plt.ylim(-max_val, max_val)
plt.axhline(0, color='black', lw=1, alpha=0.5)
plt.axvline(0, color='black', lw=1, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Loadings (Top Genes)")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
#Biplot
plt.figure(figsize=(12, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=labels,
                palette='tab10', s=150, edgecolor='black', alpha=0.9)
# Get Top Genes
loadings_dist = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
top_indices = np.argsort(loadings_dist)[-10:]
top_indices = top_indices[np.argsort(loadings[top_indices, 1])]
#top_n = 10
#top_pc1 = np.argsort(np.abs(loadings[:, 0]))[-top_n:]
#top_pc2 = np.argsort(np.abs(loadings[:, 1]))[-top_n:]
#top_indices = np.unique(np.concatenate([top_pc1, top_pc2]))

multiplier = np.max(np.abs(X_pca[:, 0])) / np.max(np.abs(loadings[top_indices, 0])) * 0.75
for i, idx in enumerate(top_indices):
    x_arrow = loadings[idx, 0] * multiplier
    y_arrow = loadings[idx, 1] * multiplier
    y_range = np.max(X_pca[:, 1]) - np.min(X_pca[:, 1])
    x_range = np.max(X_pca[:, 0]) - np.min(X_pca[:, 0])
    plt.arrow(0, 0, x_arrow, y_arrow, color='red', alpha=0.4,  head_width=0.02 * x_range,  length_includes_head=True)
    ha = 'left' if x_arrow > 0 else 'right'
    y_text_offset = y_arrow + (i - len(top_indices)/2) * 0.02 * y_range
    plt.text(x_arrow * 1.05, y_text_offset, gene_names[idx],
             color='darkred', fontsize=10, fontweight='bold',
             ha=ha, va='center')
plt.axhline(0, color='black', lw=1, alpha=0.2)
plt.axvline(0, color='black', lw=1, alpha=0.2)
plt.title("PCA Biplot: Pan Cancer Atlas data", fontsize=15)
plt.xlabel(f"PC1 ({exp_var[0]:.1f}%)")
plt.ylabel(f"PC2 ({exp_var[1]:.1f}%)")
plt.legend(title="Timepoint", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.margins(x=0.2)
plt.grid(alpha=0.1)
plt.show()

In [ ]:
#tSNE on dimension-reduced data from PCA
pca_for_tSNE = PCA(n_components = 80)

X_pca_for_tSNE = pca_for_tSNE.fit_transform(X)
perp_value = 250
tsne = TSNE(n_components = 2, perplexity = perp_value, random_state = 42, init='pca', learning_rate = 'auto')
X_tsne = tsne.fit_transform(X_pca_for_tSNE)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=labels, palette='tab10', s=100, edgecolor='black')
plt.title(f"t-SNE Plot (Perplexity = {perp_value})")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(title="Timepoint", bbox_to_anchor=(1.25, 1))
plt.grid(alpha=0.2)
plt.show()